# Project 5: Inventory Demand Forecasting
## Notebook 1: Dataset Collection, Data Cleaning & Preprocessing

**Course:** CSAI - Machine Learning Evaluation (Phase 1)  
**Dataset:** Walmart M5 Forecasting - Accuracy  
**Module Focus:**
- Strict **1-to-1 cleaning** mapping from `data/raw/` (5 files) to `data/processed/` (5 files).
- Schema integrity, completeness, and non-negativity verification.
- Feature engineering: Calendar cyclical trigonometric encodings, event labeling, and price discounting.
- Exporting pure **CSV** format for all 5 cleaned datasets.


In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_quality import M5DataQualityAuditor
from src.data_cleaning import M5DataCleaner

print("Data Cleaning environment initialized.")


Data Cleaning environment initialized.


### 1. Raw Data Ingestion (`data/raw/`)
We inspect the 5 raw files in `data/raw/`:
1. `calendar.csv`: 1,969 days of calendar events and SNAP program dates.
2. `sell_prices.csv`: 6,841,121 weekly price observations per store and item.
3. `sales_train_validation.csv`: 30,490 time series over 1,913 days (58.3M records).
4. `sales_train_evaluation.csv`: 30,490 time series extended to 1,941 days.
5. `sample_submission.csv`: 60,980 rows formatting the 28-day forecasting horizon (F1 to F28).


In [2]:
raw_dir = Path("../data/raw")

raw_files = [
    "calendar.csv",
    "sell_prices.csv",
    "sales_train_validation.csv",
    "sales_train_evaluation.csv",
    "sample_submission.csv"
]
for f in raw_files:
    p = raw_dir / f
    if p.exists():
        sz = p.stat().st_size / (1024 * 1024)
        print(f"Raw File: {f:<30} ({sz:>6.2f} MB)")


Raw File: calendar.csv                   (  0.10 MB)
Raw File: sell_prices.csv                (193.97 MB)
Raw File: sales_train_validation.csv     (114.45 MB)
Raw File: sales_train_evaluation.csv     (116.10 MB)
Raw File: sample_submission.csv          (  4.99 MB)


### 2. Comprehensive Data Quality Audit
Evaluating missingness, non-negativity, and schema continuity.


In [3]:
from src.data_loader import load_calendar, load_sell_prices, load_sales_train

auditor = M5DataQualityAuditor(raw_dir="../data/raw", reports_dir="../reports")
cal_audit = auditor.audit_calendar(load_calendar(str(raw_dir / "calendar.csv")))
price_audit = auditor.audit_sell_prices(load_sell_prices(str(raw_dir / "sell_prices.csv")))
sales_audit = auditor.audit_sales_train(load_sales_train(str(raw_dir / "sales_train_validation.csv")))

print(f"Total Observations Audited: {sales_audit['total_observations']:,}")
print(f"Sales Missing Values:       {sales_audit['data_cleanliness']['missing_values']}")
print(f"Negative Sales Detected:    {sales_audit['data_cleanliness']['negative_sales']}")
print(f"Negative Prices Detected:   {price_audit['anomalies']['negative_prices']}")


Total Observations Audited: 58,327,370
Sales Missing Values:       0
Negative Sales Detected:    0
Negative Prices Detected:   0


### 3. Executing 1-to-1 Data Cleaning (5 Raw -> 5 Cleaned Files)
We clean each of the 5 raw files and export the exact 5 cleaned CSV files into `data/processed/`:
1. `calendar.csv` -> `cleaned_calendar.csv`
2. `sell_prices.csv` -> `cleaned_sell_prices.csv`
3. `sales_train_validation.csv` -> `cleaned_sales_train_validation.csv`
4. `sales_train_evaluation.csv` -> `cleaned_sales_train_evaluation.csv`
5. `sample_submission.csv` -> `cleaned_sample_submission.csv`


In [4]:
cleaner = M5DataCleaner(raw_dir="../data/raw", processed_dir="../data/processed")
cleaner.run_cleaning_pipeline()


  M5 DATA CLEANING: 5 RAW FILES -> 5 CLEANED FILES
[1/5 CLEANING] Processing calendar.csv...
[SAVED] cleaned_calendar.csv (1,969 rows)
[2/5 CLEANING] Processing sell_prices.csv...


[SAVED] cleaned_sell_prices.csv (6,841,121 rows)
[3/5 CLEANING] Processing sales_train_validation.csv...


[SAVED] cleaned_sales_train_validation.csv (30,490 rows, 1913 days)
[4/5 CLEANING] Processing sales_train_evaluation.csv...


[SAVED] cleaned_sales_train_evaluation.csv (30,490 rows, 1941 days)
[5/5 CLEANING] Processing sample_submission.csv...


[SAVED] cleaned_sample_submission.csv (60,980 rows)

[DONE] All 5 cleaned files successfully created in 38.47 seconds.
Directory: /Users/dakarapuhemamdharnath/Desktop/AMLPROJECT/data/processed


### 4. Verification of Processed Output (`data/processed/`)
Confirming exactly 5 cleaned CSV files corresponding to the 5 raw files.


In [5]:
proc_dir = Path("../data/processed")
cleaned_files = sorted([f for f in os.listdir(proc_dir) if f.endswith('.csv')])

print(f"Total Cleaned CSV Files in data/processed/: {len(cleaned_files)}\n")
for i, f in enumerate(cleaned_files, 1):
    sz = (proc_dir / f).stat().st_size / (1024 * 1024)
    df_head = pd.read_csv(proc_dir / f, nrows=2)
    print(f"{i}. {f:<38} ({sz:>6.2f} MB, {len(df_head.columns)} cols)")


Total Cleaned CSV Files in data/processed/: 5

1. cleaned_calendar.csv                   (  0.21 MB, 22 cols)
2. cleaned_sales_train_evaluation.csv     (116.10 MB, 1947 cols)
3. cleaned_sales_train_validation.csv     (114.45 MB, 1919 cols)
4. cleaned_sample_submission.csv          (  4.99 MB, 29 cols)
5. cleaned_sell_prices.csv                (274.28 MB, 7 cols)
